### Bücherregal – Lösung Schritt für Schritt

*36. Bundeswettbewerb Informatik (2017), 1. Runde, Junioraufgabe 1*

Laura hat ein langes Regalbrett an ihrer Wand. Sie möchte es hübscher einräumen. Außer ihren Büchern sollen auf dem Regal
auch vier Deko-Figuren stehen. Sie überlegt sich, dass sie das Regal mit den Figuren in fünf Abschnitte unterteilen kann.
Diese Abschnitte möchte sie mit ihren Büchern füllen. Wie breit die Abschnitte sind, ist ihr egal. Sie möchte aber, dass
die Bücher im selben Abschnitt alle ungefähr gleich hoch sind. Dazu möchte sie sicherstellen, dass der Höhenunterschied
zwischen dem größten und dem kleinsten Buch im selben Abschnitt höchstens 3 cm ist. Ob die Bücher in einem Abschnitt
inhaltlich zusammenpassen, spielt für Laura keine Rolle. Nach dem Umräumen darf kein Buch übrig bleiben.

#### Junioraufgabe 1

Schreibe ein Programm, das berechnet, ob eine Aufstellung der Bücher und Figuren nach Lauras Wünschen möglich ist. Dein
Programm soll auch für größere Anzahlen von Deko-Figuren funktionieren. Dazu bekommt es als Eingabe die Anzahl der Figuren,
die Anzahl der Bücher und für jedes Buch dessen Höhe (in cm). Du kannst davon ausgehen, dass alle Bücher und die Figuren
zusammen auf das Regal passen.

Gib zunächst aus, ob eine Aufstellung möglich ist. Wenn ja, gib für jeden Abschnitt die Höhen der Bücher aus.

Wende dein Programm auf die Beispiele an, die du auf den BwInf-Webseiten findest.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `buecherregal1.txt` bis `buecherregal6.txt`. Jede Datei hat diesen Aufbau:

- 1. Zeile: Anzahl der Figuren
- 2. Zeile: Anzahl der Bücher
- danach je Zeile die Höhe eines Buches

**Achtung:** Laut Lösungshinweisen sind die Höhen in den Dateien in **Millimetern** angegeben (zum Beispiel `168`), nicht in Zentimetern wie im
Aufgabentext. Die 3 cm sind also **30 mm**.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Sortieren:** Bücher ähnlicher Höhe stehen nach dem Sortieren nebeneinander. Python kann das mit `sorted(liste)`.
- **Gieriges Verfahren (Greedy):** Man nimmt immer das kleinste noch nicht eingeräumte Buch und packt in denselben Abschnitt so viele Bücher wie möglich (alle, die höchstens
  30 mm höher sind). Erst dann beginnt man den nächsten Abschnitt. Das ist optimal: Es entstehen so wenige Abschnitte wie überhaupt möglich.
- **Lineare Suche in einer sortierten Liste:** Man geht die sortierte Liste einmal von vorn nach hinten durch.
- **Zählen und Vergleichen:** Es werden `D + 1` Abschnitte gebraucht, wenn es `D` Figuren gibt (jede Figur trennt zwei Abschnitte). Man vergleicht die Anzahl der Abschnitte mit `D + 1`.

**Nützliche Funktionen** (Bibliotheken braucht man keine)
- `sorted`, `min`, `max`, `len`, `append`, Listen in Listen

## Die Idee

1. Alle Bücherhöhen aus der Datei lesen und **sortieren**.
2. Das kleinste Buch beginnt den ersten Abschnitt. Alle folgenden Bücher, die höchstens 30 mm höher sind als dieses **erste** Buch des Abschnitts, kommen in denselben Abschnitt.
   Weil die Liste sortiert ist, ist das erste Buch des Abschnitts das kleinste und das zuletzt hinzugefügte das größte.
3. Das erste Buch, das mehr als 30 mm höher ist, beginnt den nächsten Abschnitt. So geht es weiter, bis alle Bücher einsortiert sind.
4. Es entstehen `a` Abschnitte. Dafür braucht man `a − 1` Figuren. Sind es höchstens `D` (die vorhandene Anzahl), ist die Aufstellung möglich.

**Warum ist das optimal?** Angenommen, es gäbe eine Aufteilung mit weniger Abschnitten. Der erste Abschnitt kann höchstens alle Bücher bis zum kleinsten Buch + 30 mm enthalten. Das
gierige Verfahren nimmt genau diese. Jeder spätere Abschnitt beginnt dadurch so weit rechts wie möglich. Wer weniger nimmt, kommt nie weiter als das gierige Verfahren.

Bleiben Figuren übrig (das Verfahren braucht weniger als `D`), kann man sie irgendwo mitten in einen Abschnitt stellen: Die Bücher dort sind trotzdem höchstens 30 mm unterschiedlich.

## Schritt 1: Die Datei einlesen

Alle Zahlen der Datei stehen untereinander. Mit `split()` zerlegen wir sie in eine Liste von Wörtern und wandeln sie mit `int(...)` in Zahlen um.
Die erste Zahl ist die Anzahl der Figuren, die zweite die Anzahl der Bücher, danach folgen die Höhen.

In [ ]:
def lies_regal(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zahlen = f.read().split()
    figuren = int(zahlen[0])
    anzahl_buecher = int(zahlen[1])
    hoehen = []
    for i in range(anzahl_buecher):
        hoehen.append(int(zahlen[2 + i]))
    return figuren, hoehen


figuren, hoehen = lies_regal('beispieldaten/buecherregal1.txt')
print('Figuren:', figuren)
print('Bücherhöhen in mm:', hoehen)

Figuren: 4
Bücherhöhen in mm: [168, 170, 202, 211, 229, 233, 254, 260, 272, 306, 307]


## Schritt 2: Abschnitte bilden

Wir sortieren die Höhen und gehen sie der Reihe nach durch. Die Variable `abschnitte` ist eine Liste von Abschnitten, jeder Abschnitt wieder eine Liste von Höhen.
Für jedes Buch fragen wir: Ist es höchstens 30 mm höher als das **erste** Buch des aktuellen Abschnitts (`abschnitte[-1][0]`, der letzte Abschnitt, dessen erstes Buch)?
Dann kommt es dazu, sonst beginnt ein neuer Abschnitt.

In [ ]:
MAX_UNTERSCHIED = 30                      # 3 cm = 30 mm


def bilde_abschnitte(hoehen):
    abschnitte = []
    for hoehe in sorted(hoehen):
        if len(abschnitte) > 0 and hoehe - abschnitte[-1][0] <= MAX_UNTERSCHIED:
            abschnitte[-1].append(hoehe)        # passt noch in den letzten Abschnitt
        else:
            abschnitte.append([hoehe])          # neuer Abschnitt mit diesem Buch
    return abschnitte


abschnitte = bilde_abschnitte(hoehen)
for abschnitt in abschnitte:
    print(abschnitt)

[168, 170]
[202, 211, 229]
[233, 254, 260]
[272]
[306, 307]


Die elf Bücher von `buecherregal1.txt` ergeben fünf Abschnitte (nach den Höhen 168 bis 306 mm). Dafür sind vier Figuren nötig. Es sind genau vier vorhanden: Die Aufstellung ist möglich.

## Schritt 3: Prüfen und ausgeben

Wir sagen, ob es geht, und geben die Abschnitte aus, getrennt durch eine `Figur`. Bei langen Abschnitten (die großen Beispiele haben 100 Bücher) geben wir nur
die Anzahl, das kleinste und das größte Buch an. Das genügt laut Hinweisen.

Außerdem schreiben wir eine **Prüffunktion**: Sie kontrolliert, dass kein Buch verloren ging, dass in jedem Abschnitt der Unterschied höchstens 30 mm ist und dass nicht mehr
Abschnitte als `figuren + 1` vorkommen.

In [ ]:
def ist_korrekt(hoehen, abschnitte, figuren):
    alle = []
    for abschnitt in abschnitte:
        if max(abschnitt) - min(abschnitt) > MAX_UNTERSCHIED:
            return False
        alle = alle + abschnitt
    return sorted(alle) == sorted(hoehen) and len(abschnitte) <= figuren + 1


def bearbeite(dateiname):
    figuren, hoehen = lies_regal(dateiname)
    abschnitte = bilde_abschnitte(hoehen)
    noetig = len(abschnitte) - 1                # so viele Figuren trennen die Abschnitte
    moeglich = noetig <= figuren
    print(f'{dateiname}: {len(hoehen)} Bücher, {figuren} Figuren vorhanden, {noetig} Figuren benötigt')
    if not moeglich:
        print('  Unmöglich: Es sind zu wenige Figuren da.')
        print()
        return
    print('  Möglich. ', end='')
    if figuren > noetig:
        print(f'({figuren - noetig} Figuren bleiben übrig und können mitten in einen Abschnitt gestellt werden.)')
    else:
        print()
    for nr in range(len(abschnitte)):
        abschnitt = abschnitte[nr]
        if nr > 0:
            print('  Figur')
        if len(abschnitt) <= 8:
            print('  ', abschnitt)
        else:
            print(f'   {len(abschnitt)} Bücher von {abschnitt[0]} bis {abschnitt[-1]} mm')
    print('  Prüfung bestanden:', ist_korrekt(hoehen, abschnitte, figuren))
    print()


bearbeite('beispieldaten/buecherregal1.txt')

beispieldaten/buecherregal1.txt: 11 Bücher, 4 Figuren vorhanden, 4 Figuren benötigt
  Möglich. 
   [168, 170]
  Figur
   [202, 211, 229]
  Figur
   [233, 254, 260]
  Figur
   [272]
  Figur
   [306, 307]
  Prüfung bestanden: True



## Schritt 4: Alle Beispiele

In [ ]:
for nr in range(1, 7):
    bearbeite(f'beispieldaten/buecherregal{nr}.txt')

beispieldaten/buecherregal1.txt: 11 Bücher, 4 Figuren vorhanden, 4 Figuren benötigt
  Möglich. 
   [168, 170]
  Figur
   [202, 211, 229]
  Figur
   [233, 254, 260]
  Figur
   [272]
  Figur
   [306, 307]
  Prüfung bestanden: True

beispieldaten/buecherregal2.txt: 7 Bücher, 2 Figuren vorhanden, 2 Figuren benötigt
  Möglich. 
   [169, 175]
  Figur
   [203, 209, 210, 229]
  Figur
   [235]
  Prüfung bestanden: True

beispieldaten/buecherregal3.txt: 7 Bücher, 2 Figuren vorhanden, 3 Figuren benötigt
  Unmöglich: Es sind zu wenige Figuren da.

beispieldaten/buecherregal4.txt: 100 Bücher, 4 Figuren vorhanden, 4 Figuren benötigt
  Möglich. 
   31 Bücher von 160 bis 190 mm
  Figur
   21 Bücher von 196 bis 225 mm
  Figur
   20 Bücher von 233 bis 261 mm
  Figur
   22 Bücher von 264 bis 293 mm
  Figur
   [295, 296, 300, 301, 303, 304]
  Prüfung bestanden: True

beispieldaten/buecherregal5.txt: 100 Bücher, 3 Figuren vorhanden, 3 Figuren benötigt
  Möglich. 
   50 Bücher von 160 bis 180 mm
  Figur
   

Das Ergebnis stimmt mit der Tabelle der offiziellen Lösungshinweise überein:

| Beispiel | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| Figuren vorhanden | 4 | 2 | 2 | 4 | 3 | 4 |
| Figuren benötigt | 4 | 2 | 3 | 4 | 3 | 5 |
| Ergebnis | möglich | möglich | unmöglich | möglich | möglich | unmöglich |

Bei den unmöglichen Fällen hilft auch mehr Probieren nicht: Das gierige Verfahren findet **immer** die kleinstmögliche Zahl von Abschnitten. Braucht es mehr als `figuren + 1`,
geht es mit keiner anderen Aufteilung.

## Für Neugierige: Stimmt die Optimalität wirklich?

Wir prüfen die Behauptung „gierig ist optimal“ an vielen kleinen Zufallsfällen. Dazu probieren wir bei einer sortierten Liste **alle** Möglichkeiten durch, Trennstellen zu setzen
(`itertools.combinations(positionen, k)` liefert alle Auswahlen von `k` Positionen), und suchen die kleinste Zahl von Abschnitten, bei der jeder Abschnitt den Höhenunterschied einhält.
Optimale Aufteilungen bestehen immer aus Abschnitten, die in der sortierten Reihenfolge direkt nebeneinander liegen (Vertauschen von Büchern zwischen Abschnitten bringt nichts).
Der Test vergleicht die beiden Verfahren.

In [ ]:
import itertools
import random


def wenigste_abschnitte_brute_force(hoehen):
    s = sorted(hoehen)
    n = len(s)
    for abschnitte in range(1, n + 1):
        # Trennstellen: Positionen 1 bis n-1, wir wählen abschnitte-1 davon
        for stellen in itertools.combinations(range(1, n), abschnitte - 1):
            grenzen = [0] + list(stellen) + [n]
            gut = True
            for i in range(len(grenzen) - 1):
                teil = s[grenzen[i]:grenzen[i + 1]]
                if max(teil) - min(teil) > MAX_UNTERSCHIED:
                    gut = False
            if gut:
                return abschnitte
    return n


random.seed(3)
alles_gleich = True
for versuch in range(300):
    liste = []
    for i in range(random.randint(1, 10)):
        liste.append(random.randint(100, 300))
    if len(bilde_abschnitte(liste)) != wenigste_abschnitte_brute_force(liste):
        alles_gleich = False
print('Gierig und Brute Force finden immer gleich viele Abschnitte:', alles_gleich)

Gierig und Brute Force finden immer gleich viele Abschnitte: True
